# RETFoundGreen Diagnosis Model Training (Single-Image)

Trains the single-image diabetic retinopathy diagnosis model (RETFoundGreen backbone). This is the single-image baseline and also the checkpoint source used downstream by the cascade and multi-image fusion pipelines.

In [ ]:
import sys, os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

from model import UnifiedBackbone
from fundus_dataset import FundusDataset
from diagnosis_train_eval import train_model, validate

## Configuration

In [ ]:
# ── Run configuration ─────────────────────────────────────────────────────
RUN_ID     = 4
DATASET    = "BRSET"          # "BRSET" or "mBRSET"
MODEL_NAME = "retfound_green"

# ── Data paths (edit for your environment) ───────────────────────────────
BRSET_DATA_DIR  = r"C:\Users\preet\Documents\BRSET\data"
MBRSET_DATA_DIR = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\data"
BRSET_IMG_ROOT  = r"C:\Users\preet\Documents\BRSET\data\resized_fundus_photos"
MBRSET_IMG_ROOT = r"C:\Users\preet\Documents\mBRSET\mbrset-a-mobile-brazilian-retinal-dataset-1.0\images"

# Checkpoints are saved as f"{CHECKPOINT_PREFIX}_img_diagnosis_model_top{rank}_BA_{ba}.pth"
CHECKPOINT_PREFIX = f"run{RUN_ID}_{DATASET}_single_"

## Load Data

In [ ]:
if DATASET == "mBRSET":
    train_df = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_train_full.pkl"))
    val_df   = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_val_full.pkl"))
    test_df  = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_test_full.pkl"))
    img_root = MBRSET_IMG_ROOT

elif DATASET == "BRSET":
    train_df = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_train_524.pkl"))
    val_df   = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_val_524.pkl"))
    test_df  = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_test_524.pkl"))
    img_root = BRSET_IMG_ROOT
    # Val and test: one image per patient (laterality == 0)
    val_df  = val_df[val_df["laterality"] == 0]
    test_df = test_df[test_df["laterality"] == 0]

for df in (train_df, val_df, test_df):
    df.rename(columns={"patient_id": "patient"}, inplace=True)
    df.dropna(subset=["final_icdr"], inplace=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Train disease prevalence: {(train_df['final_icdr'] > 0).mean():.3f}")

## Transforms

In [ ]:
if MODEL_NAME == "retfound_green":
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
else:
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = A.Compose([
    A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(392, 392),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

## Datasets & Loaders

In [ ]:
train_ds = FundusDataset(train_df, img_root, high_quality_tf=train_tf, low_quality_tf=train_tf, label_col="final_icdr")
val_ds   = FundusDataset(val_df,   img_root, high_quality_tf=val_tf,   low_quality_tf=val_tf,   label_col="final_icdr")
test_ds  = FundusDataset(test_df,  img_root, high_quality_tf=val_tf,   low_quality_tf=val_tf,   label_col="final_icdr")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=0)

## Train Diagnosis Model

In [ ]:
def compute_class_weights(df, label_col="final_icdr"):
    labels  = (df[label_col] > 0).astype(int).values
    counts  = np.bincount(labels)
    weights = 1.0 / counts
    return torch.tensor(weights / weights.sum(), dtype=torch.float32)

device = "cuda"
model  = UnifiedBackbone(model_name=MODEL_NAME)

loss_fn   = nn.CrossEntropyLoss(weight=compute_class_weights(train_df).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)

best_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=20,
    patience=5,
    str_prefix=CHECKPOINT_PREFIX,
)

## Evaluate

In [ ]:
def report(split_name, loss, metrics):
    print(f"{split_name}  loss={loss:.4f}  BA={metrics['ba']:.4f}  "
          f"accuracy={metrics['accuracy']:.4f}  AUC={metrics['roc_auc']:.4f}  "
          f"AUPRC={metrics['auprc']:.4f}  F1={metrics['f1']:.4f}")
    print(f"Confusion matrix:\n{metrics['conf_matrix']}\n")

val_loss, val_metrics, *_   = validate(best_model, val_loader,  loss_fn, device)
test_loss, test_metrics, *_ = validate(best_model, test_loader, loss_fn, device)

report("Val ", val_loss, val_metrics)
report("Test", test_loss, test_metrics)